
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Demo - Tagging & Reproducible Agents

This demonstration explores advanced MLflow tracing techniques for building production-ready AI agents. Building on foundational tracing concepts, we'll learn how to implement tagging strategies for better trace management and create reproducible agents through Unity Catalog registration.

## Learning Objectives

By the end of this demonstration, you will be able to:

- Implement MLflow tagging strategies to organize and manage agent traces effectively
- Create custom trace functions with proper validation and error handling
- Log agent models to MLflow with appropriate configuration and dependencies
- Register agent models to Unity Catalog for governance and reproducibility
- Deploy and inference agents from both MLflow and Unity Catalog registries

## A. Environment Setup

### A1. Compute Requirements

**🚨 REQUIRED - SELECT SERVERLESS COMPUTE**

This course has been configured to run on Serverless compute. While classic compute may also work, testing has been performed on serverless.

**This demo was tested using version 5 of Serverless compute.** To ensure that you are using the correct version of Serverless, please [see this documentation on viewing and changing your notebook's Serverless version.](https://docs.databricks.com/aws/en/compute/serverless/dependencies)

### A2. Install Dependencies

As part of the workspace setup, several Python libraries will need to be installed. Run the next cell to do so.

In [0]:
%run ../Includes/Classroom-Setup-3.2

### A3. Inspect the Airbnb Dataset

As part of the classroom setup, the Airbnb dataset has been processed and stored as a Delta table within Unity Catalog. Run the next cell to query the first few rows of the dataset.

In [0]:
df = spark.read.table('sf_airbnb_listings')
display(df.limit(5))

### A4. Initialize MLflow Autologging

MLflow's autologging automatically captures traces for supported frameworks like LangChain. When enabled, it records inputs, outputs, parameters, and metrics without requiring manual instrumentation.

In [0]:
import mlflow

mlflow.langchain.autolog()

### A5. Define Experiment Locations

We'll create an experiment using the **default location** for the artifact and setting the experiment location to your **User** folder.

> Workspace MLflow experiments cannot be created in Git folders; use a non-Git workspace experiment. Notebooks in Git folders can still log to their notebook experiment, but with management limitations.

In [0]:
# Get username
username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()

experiment_name = f"/Workspace/Users/{username}/single_agents_demo3" 

### A6. Load the Agent

We'll initialize our agent with MLflow autologging enabled and configure the experiment for trace collection. As a part of the classroom setup, we have created a config file called `demo_agent1_config.json` using a helper function (`create_demo_agent_config`) defined in the setup script. 

> This course does not go into the details of how to build an agent, but rather assumes you have some experience with the concept of an agent.

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import demo_agent1
agent = demo_agent1.DatabricksAgent(
    catalog_name=catalog_name,
    schema_name=schema_name
)

## B. Tagging Agents with MLflow

Following the last example from the previous demo, we will look at custom tracing that validates the user's input before sending the request to the LLM.

### B1. Why Use Tags?

Tags make trace management easier by allowing you to:

- **Manage Sessions:** Group traces by user conversations or interaction sessions
- **Track Environments:** Differentiate between development, staging, and production runs
- **Version Models:** Identify which model version produced each trace
- **Add User Context:** Link traces to specific users or audience segments
- **Monitor Performance:** Label traces based on latency or throughput metrics
- **Support A/B Testing:** Mark traces from different experimental variants for comparison

We will focus on active traces in this demonstration, but you can read more about different tag types [here](https://mlflow.org/docs/3.2.0/genai/tracing/attach-tags/#when-to-use-trace-tags).

### B2. Setting Up Tags

Let's set up our tags outside the `@mlflow.trace` decorator code. Below is an example of a set of tags we can pass through to the validation function that follows. The `tags` object is made up of key-value pairs.

In [0]:
tags = {
        "component": "input_validation",
        "stage": "preprocessing",
        "span_scope": "tool_function",
        "env": "dev",
        "trace_version": "v1.0.0",
        "input_type": "question"
    }

### B3. Import Required Libraries

Let's bring in the necessary libraries for our tracing implementation. Recall from the previous demo that, for example, `span_type=SpanType.TOOL` classifies a function as a tool span in the trace UI, making it easier to identify different types of operations (like `FUNC`, `TOOL`, `CHAIN`, etc.) when reviewing traces. You can read more about span types [here](https://docs.databricks.com/aws/en/mlflow3/genai/tracing/data-model#span-types).

In [0]:
from mlflow.entities import SpanType

### B4. Create Tagged Validation Function

Now we will pass the tags through to the function using `mlflow.update_current_trace(tags)`. This will occur _inside_ the function definition. Again, keep in mind this is for an _active trace_.

In [0]:
@mlflow.trace(
    span_type=SpanType.TOOL, 
    name="Validate Input"
)
def validate_input(question: str, tags: dict, min_length: int = 5):
    """Check if the user's question meets basic requirements"""
    
    mlflow.update_current_trace(tags)

    if len(question) < min_length:
        return {
            "valid": False,
            "error": f"Question too short (minimum {min_length} characters)"
        }
    if question.strip() == "":
        return {
            "valid": False,
            "error": "Question cannot be empty"
        }
    return {
        "valid": True,
        "cleaned_question": question.strip()
    }

In [0]:
@mlflow.trace(
    name="Call LLM",
    span_type=SpanType.CHAT_MODEL
)
def call_llm(question: str):
    return agent.ask(question)

In [0]:
@mlflow.trace(name="Process Question")
def process_question(user_input: str, tags: dict):
    """Main function that validates input and calls LLM"""
    # Step 1: Validate the input
    validation_result = validate_input(user_input, tags)
    
    # Step 2: If valid, call the LLM
    cleaned = validation_result["cleaned_question"]
    llm_response = call_llm(cleaned)
    
    return llm_response

### B5. Test Tagged Tracing

Define the prompt and test our tagged tracing implementation.

In [0]:
prompt = "Can you tell me the average for Mission?"

Call `process_question` using the prompt and tags defined above. Notice the MLflow Trace UI now has an additional output called **Tags**.

**Instructions:**
1. Run the next cell
2. Click on **Tags** to view the tags we set up with the variable `tags` defined above

In [0]:
# Test with a valid question
result = process_question(prompt, tags)

## C. Reproducible Agents

Though we have an agent's trace and tags for `process_question`, we don't yet have code that can be shared with other teams for testing and further development, not to mention proper governance of our agent. We'll accomplish this in two steps:
1. Log the agent with MLflow
2. Register the model to Unity Catalog

We'll start by using a custom function that will build the `demo_agent_config.json` file. This needs to be specific to the **catalog** and **schema** you are using in this lab environment. In practice, making this dynamic or static depends on your use case.

### C1. Read Agent Configuration

The following prints a configuration file that defines your agent's settings and tools.

In [0]:
import json
# Read in Agent JSON config file 
with open('demo_agent2_config.json', 'r') as f:
    agent_config = json.load(f)
print(agent_config)

In [0]:
%reload_ext autoreload
%autoreload 2

In [0]:
from demo_agent2 import AGENT

### C2. Logging the Agent to MLflow

We'll log our agent as a PyFunc model to MLflow, including all necessary dependencies and configuration files using `with mlflow.start_run`. Additionally, we will add tags (`tags_to_register`) that designate the agent's framework (`openai`), the stage of development (`dev`), and the version number (`1`) for discoverability by other teams. We do this using `mlflow.set_tags(tags)`.

**NOTE:** In the next cell we are bringing in additional modules that will be needed for logging with MLflow:
1. The `pkg_resources` module and it is going to import the `get_distribution` function. This function is used to query the installed version of a Python package at runtime (in this case `databricks-connect`)
2. The `mlflow.models.resources` module will enable automatic authentication passthrough for specified resources defined by `resources`. You can read more [here](https://docs.databricks.com/aws/en/generative-ai/agent-framework/agent-authentication#implement-automatic-authentication-passthrough)

In [0]:
from importlib.metadata import version
from mlflow.models.resources import (
    DatabricksFunction,
    DatabricksTable,
    DatabricksServingEndpoint
)

input_example = {
    "input": [
        {
            "role": "user",
            "content": prompt
        }
    ]
}

model_name = "tagging-and-reproducible-agents"

tags_to_register = {
    "framework": "openai",
    "stage": "dev",
    "version": "1"
}

resources = [
    DatabricksFunction(function_name=agent_config["tool_list"][0]),
    DatabricksFunction(function_name=agent_config["tool_list"][1]),
    DatabricksTable(table_name=f"{catalog_name}.{schema_name}.sf_airbnb_listings"),
    DatabricksServingEndpoint(endpoint_name=agent_config["llm_endpoint"])
]

In [0]:
with mlflow.start_run():
    mlflow.set_tags(tags_to_register)
    logged_agent_info = mlflow.pyfunc.log_model(
        name=model_name,
        python_model="demo_agent2.py",
        code_paths=["demo_agent2_config.json"],
        input_example=input_example,
        pip_requirements=[
            "databricks-openai",
            "backoff",
            f"databricks-connect=={version('databricks-connect')}",
        ],
        resources=resources
    )
    model_uri = logged_agent_info.model_uri # Save the model URI to model_uri for use down below

The output above will show **View Logged Model at: <url>**. Click on the URL and see that the model has been logged to MLflow, but not registered. Notice we have configured some tags as well (you can find and edit these tags in the `demo_agent2.py` file located in the same folder as this notebook).

**Instructions:**
1. On the landing page of the URL you clicked, navigate to **Runs**
2. There, you will find a randomly named run like **tasteful-slug-677**. Click on it
3. This will take you to the overview page for the run, where you will see 5 tabs at the top of the page. Click on **Artifacts**
4. Under **Logged models artifacts**, click the dropdown menu for `tagging-and-reproducible-agents`. This contains all the various files that define the logged models with MLflow. We won't go into all the details here, but you can read more in this [documentation](https://docs.databricks.com/aws/en/mlflow/models)

### C3. Inferencing the Agent from MLflow

As part of the MLflow run, we have saved the model URI to the variable `model_uri`. Let's load our model and perform a simple inference with the agent using the logged input data (note this is the same as the `prompt` variable defined above). Read the output from this cell and see that our UC functions were indeed called and we were returned the response like _"The average listing price for Airbnb properties in the Mission neighborhood is approximately $229.76."_

In [0]:
# Load the model (pyfunc flavor)
pyfunc_model = mlflow.pyfunc.load_model(model_uri)

# The model is logged with an input example
input_data = pyfunc_model.input_example

# Verify the model with the provided input data using the logged dependencies.
result = mlflow.models.predict(
    model_uri=model_uri,
    input_data=input_data,
    env_manager="uv",
)

### C4. Registering the Agent to Unity Catalog

Now that we have our model logged with MLflow, it's time to register it to Unity Catalog. Recall we defined `model_name` when logging our model to MLflow above.

In [0]:
mlflow.set_registry_uri("databricks-uc")
UC_MODEL_NAME = f"{catalog_name}.{schema_name}.{model_name}"

# register the model to UC
uc_registered_model_info = mlflow.register_model(
    model_uri=model_uri, 
    name=UC_MODEL_NAME
)

### C5. Inferencing the Agent from Unity Catalog

Inferencing the model that's registered to UC is exactly the same as inferencing the logged agent with MLflow. We will simply need to update the URI using `mlflow.set_registry_uri("databricks-uc")` as shown in the next cell. Run the next cell and view the output.

In [0]:
import mlflow
from mlflow.types.responses import ResponsesAgentRequest

mlflow.set_registry_uri("databricks-uc")

# Load the model (pyfunc flavor)
pyfunc_model = mlflow.pyfunc.load_model(model_uri)

# The model is logged with an input example
input_data = pyfunc_model.input_example

# Verify the model with the provided input data using the logged dependencies.
result = mlflow.models.predict(
    model_uri=model_uri,
    input_data=input_data,
    env_manager="uv",
)

### C6. Exploring Unity Catalog Model Interface

Note that all traces will be logged with MLflow. However, traces for UC-registered models will appear under that model's trace in UC's UI.

**Instructions:**
Navigate to your model (located at `catalog_name.schema_name.tagging-and-reproducible-agents`) and click on the most recent version of the model. There, you will find four different tabs:

- **Overview**: here you will see any metrics that have been logged with the model, activity log, the model's signature, information about the version, active endpoints, and tags
- **Lineage**: this will show this current notebook as an upstream asset
- **Artifacts**: these are the same artifacts that are registered to MLflow
- **Traces**: these are traces captured for this particular version

## Conclusion

In this demonstration, we explored advanced techniques for building production-ready AI agents with MLflow and Unity Catalog. We learned how to:

- Implement comprehensive tagging strategies for better trace organization and management
- Create robust validation functions with proper error handling and trace annotation
- Log agent models to MLflow with complete configuration and dependency management
- Register agents to Unity Catalog for enterprise governance and reproducibility
- Deploy and inference agents from both MLflow and Unity Catalog environments

These techniques provide the foundation for building scalable, governed, and reproducible AI agent systems in production environments. The combination of MLflow tracing with Unity Catalog registration ensures that your agents are not only functional but also maintainable and compliant with enterprise standards.

## Next Steps

If you have completed the previous demo, you are ready for the hands-on lab to test your knowledge and understanding of building reproducible agents with MLflow and Unity Catalog.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>